<img src="https://www.plus2net.com/images/top2.jpg" alt="plus2net logo" style="height:40px;">

# NumPy Memory Mapping Practice Notebook

This practice notebook accompanies the [NumPy Memory Mapping Tutorial on plus2net](https://www.plus2net.com/python/numpy-memory-mapping.php).

**Objectives:**
1. Initialize large binary disk arrays using `np.memmap()` mode `'w+'`.
2. Perform chunked writes and in-place reads without loading the full matrix into RAM.
3. Apply chunked z-score normalization on disk-backed data.
4. Test read-only (`mode='r'`) and header offset parameters.
5. Flush modifications and execute self-validating assertions.

### Step 1: Initialize Large Binary Memmap File (`w+` mode)

In [ ]:
import numpy as np

filename = 'large_matrix.dat'
shape = (10000, 10000) # 100 million float64 elements (~800 MB on disk)

# Create binary memmap file on disk
fp = np.memmap(filename, dtype='float64', mode='w+', shape=shape)

# Write values chunk-by-chunk to keep memory footprint low
for i in range(0, shape[0], 1000):
    fp[i:i+1000, :] = np.random.randn(1000, shape[1])

# Flush changes to disk
fp.flush()
print('Successfully created memory-mapped array on disk!')

### Step 2: In-Place Reading and Updating (`r+` mode)

In [ ]:
# Re-open in read-write mode
new_fp = np.memmap(filename, dtype='float64', mode='r+', shape=shape)

# Slice mean calculation
slice_mean = np.mean(new_fp[500:1000, :])
print(f'Mean of slice [500:1000]: {slice_mean:.4f}')

# Update first row directly on disk
new_fp[0, :5] = [1.0, 2.0, 3.0, 4.0, 5.0]
new_fp.flush()
print('Updated first 5 elements directly on disk:', new_fp[0, :5])

### Step 3: Chunked Batch Normalization

In [ ]:
data = np.memmap(filename, dtype='float64', mode='r+', shape=shape)
chunk_size = 2000

for start in range(0, shape[0], chunk_size):
    end = start + chunk_size
    chunk = data[start:end, :]
    c_mean = np.mean(chunk)
    c_std = np.std(chunk)
    data[start:end, :] = (chunk - c_mean) / c_std

data.flush()
print('Chunked normalization completed!')

### Step 4: Read-Only Worker Access & Header Offset

In [ ]:
# Safe read-only mode
ro_data = np.memmap(filename, dtype='float64', mode='r', shape=shape)
print('Read-only sum of slice:', np.sum(ro_data[:10, :10]))

# Offset memmap (skip 800 bytes = 100 float64 elements)
offset_data = np.memmap(filename, dtype='float64', mode='r', offset=800, shape=(1000, 1000))
print('Offset view shape:', offset_data.shape)

### Step 5: Self-Validating Exercise Verification

In [ ]:
fp_ex = np.memmap('exercise_memmap.dat', dtype='float32', mode='w+', shape=(5000, 5000))
fp_ex[0, :] = 100.0
fp_ex.flush()

assert fp_ex[0, 0] == 100.0
print('Exercise verification passed successfully!')

---
### Return to Tutorial
For full technical details, visit the [NumPy Memory Mapping Tutorial on plus2net](https://www.plus2net.com/python/numpy-memory-mapping.php).